# 13장. 사전학습과 글 만들기

이야기 20만 편으로 작은 GPT를 30분 동안 학습시키고, 학습 전과 뒤의 글을 비교합니다. 학습한 모델의 어텐션을 들여다보고, 온도와 top-k로 다음 글자를 고르는 방법을 직접 써 봅니다.

- **GPU가 필요합니다.** 런타임 → 런타임 유형 변경 → T4 GPU를 고르세요.
- GPU를 받을 수 없다면 실습 13-3(30분 학습)은 건너뛰세요. 실습 13-4부터는 저자가 학습시킨 모델을 자동으로 내려받아 씁니다.
- 학습 결과는 구글 드라이브의 `MyDrive/modu_llm/` 폴더에 저장됩니다.

데이터: [Tiny-Ko-Stories](https://huggingface.co/datasets/psymon/Tiny-Ko-Stories) (psymon, CC BY 4.0) · 모델: [JosephLee/modu-llm](https://huggingface.co/JosephLee/modu-llm) (CC BY 4.0)

## 실습 13-1 준비: 이야기 20만 편

In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm-v2
import os
import torch
from torch.nn import functional as F
from modu_llm import Vocab, GPT, GPTConfig, get_batch, generate
from modu_llm import prepare_stories, load_checkpoint, get_file, train, mount_drive
from modu_llm import get_device, plot_history

device = get_device(need_gpu=True)
vocab = Vocab.build()
folder = mount_drive()
train_data, val_data = prepare_stories(200_000, vocab, folder)

## 실습 13-2 학습 한 걸음 직접 해 보기

In [ ]:
model = GPT(GPTConfig()).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)

for i in range(5):
    x, y = get_batch(train_data, 384, 32, device)
    _, loss = model(x, y)      # ① 맞히고 손실 재기
    optimizer.zero_grad()      # ② 지난 기울기 지우기
    loss.backward()            # ③ 고칠 방향 계산하기
    optimizer.step()           # ④ 숫자 고치기
    print(f"{i + 1}걸음: 손실 {loss.item():.3f}")

## 실습 13-3 30분 동안 학습하기 (GPU)

세션이 끊겨도 이 셀을 다시 실행하면 저장된 곳에서 이어서 학습합니다.

In [ ]:
ckpt_path = f"{folder}/pretrain.pt"
if os.path.exists(ckpt_path):
    model, optimizer, step, history = load_checkpoint(ckpt_path, device, optimizer_lr=5e-4)
    print(f"{step}걸음에서 이어서 학습합니다.")
else:
    model = GPT(GPTConfig()).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
    step = 0
    history = []
    print("처음부터 학습합니다.")

step, history = train(model, optimizer, train_data, val_data,
                      minutes=30, ckpt_path=ckpt_path, eval_every=250,
                      start_step=step, history=history, vocab=vocab)
print(f"학습 끝: {step}걸음 | 검증 손실 {history[-1]['val']:.3f}")

## 실습 13-4 손실 그래프 보기

드라이브에 학습한 모델(pretrain.pt)이 있으면 그것을, 없으면 저자의 모델을 불러옵니다. 이제부터는 CPU에서 계산합니다.

In [ ]:
model, _, step, history = load_checkpoint(get_file("pretrain.pt"), "cpu")
print(f"{step}걸음 학습한 모델")
plot_history(history)

## 실습 13-5 비포·애프터

In [ ]:
torch.manual_seed(0)
before = GPT(GPTConfig())
for name, m in [("학습 전", before), ("학습 뒤", model)]:
    torch.manual_seed(0)
    story = generate(m, vocab, "어느 날, ", max_new_tokens=100)
    print(f"[{name}] 어느 날, {story}")
    print()

## 실습 13-6 학습한 모델의 어텐션 들여다보기

layer(층)와 head(헤드)의 숫자를 바꿔 보세요. 층은 1~6, 헤드는 1~4입니다.

In [ ]:
from modu_llm import StoryLLM

llm = StoryLLM(model, vocab, "pretrain")
llm.attention("토끼가 숲에 가요.", layer=1, head=2)

## 실습 13-7 온도: 확률을 뾰족하게, 평평하게

In [ ]:
idx = torch.tensor([vocab.encode("어느 날, ")])
with torch.no_grad():
    logits, _ = model(idx)
last = logits[0, -1]                       # 마지막 자리의 점수
top = torch.topk(last, 5).indices          # 점수 상위 5글자
for t in [0.5, 1.0, 2.0]:
    probs = F.softmax(last / t, dim=-1)    # 온도로 나눈 뒤 확률로
    line = f"온도 {t}:"
    for i in top.tolist():
        line = line + f"  {vocab.chars[i]} {probs[i].item() * 100:.1f}%"
    print(line)

## 실습 13-8 다음 글자 고르기를 직접 써 보기

In [ ]:
def write(prompt, length=80, temperature=1.0, top_k=None):
    ids = vocab.encode(prompt)
    for i in range(length):
        x = torch.tensor([ids[-384:]])                  # 마지막 384글자만
        with torch.no_grad():
            logits, _ = model(x)
        last = logits[0, -1] / temperature             # ① 온도로 나누기
        if top_k is not None:
            kth = torch.topk(last, top_k).values[-1]
            last = last.masked_fill(last < kth, float("-inf"))   # ② k등 아래는 가리기
        probs = F.softmax(last, dim=-1)
        next_id = torch.multinomial(probs, 1).item()   # ③ 확률대로 하나 뽑기
        ids.append(next_id)                            # ④ 붙이기
    return vocab.decode(ids)

for t, k in [(0.5, None), (1.5, None), (1.5, 10)]:
    torch.manual_seed(0)
    print(f"[온도 {t}, top_k {k}]", write("어느 날, ", temperature=t, top_k=k))
    print()